# 10 - A1: LFCC + GMM (official ASVspoof 2019 B02 baseline)

**Question this notebook answers:** does the official B02 recipe -- 60-dim LFCC (20 static + delta + delta-delta), two 512-component diagonal GMMs, mean-per-frame log-likelihood-ratio scoring -- separate bona fide from spoof speech on ASVspoof 2019 LA, and how does our reproduction compare (never directly, only side by side with a `literature`-tagged reference) to the published B02 eval numbers (t-DCF 0.2116, EER 8.09%, arXiv:1904.05441 Table 1)?

**Every B02 choice this notebook relies on** is documented and sourced in `research/knowledge/methods/b02-lfcc-gmm.md` (read directly from the official baseline code) and `src/audio_df/lfcc_b02.py`'s module docstring:

- 20 ms Hamming window, 10 ms hop (50% overlap), 512-pt FFT.
- 20 linear triangular filters spanning **0-8000 Hz** (not 30 Hz-8 kHz -- an earlier, unsourced belief the research note corrected).
- Pre-emphasis 0.97, `log10` (not natural log), orthonormal DCT-II, no CMVN.
- 20 static (19 + 0th cepstral coefficient) + 20 delta + 20 delta-delta = 60 dims. Delta/delta-delta use the official `deltas.m`'s exact FIR filter (`official_deltas`, transcribed from `filter([hlen:-1:-hlen], 1, xx)` with edge-replicated padding) -- for w=3 this is `x[t+1]-x[t-1]` with **no division by 2**, unlike `features.add_deltas`'s normalised `np.gradient`.
- Two independent 512-component diagonal-covariance GMMs (bona fide, spoof), trained on the TRAIN partition only, random-init EM, 10 iterations.
- Score = mean per-frame log-likelihood-ratio (bona fide GMM minus spoof GMM) over the **whole utterance** -- A1 does not use this project's usual 4 s crop.

**Notebook rule (tasks/CONVENTIONS.md):** this notebook holds questions, plots and interpretation only; every computation calls into `src/audio_df`.

**Do not run the actual train/score commands from inside this notebook.** Cell 4 prints the exact terminal commands; run them in a terminal (ideally under `caffeinate -i`) so a multi-hour job survives the notebook's kernel restarting or the laptop trying to sleep.


## Cell 1 - config, resource limits, current memory status

Loads `configs/a1_lfcc_gmm.yaml` and `configs/resources.yaml`, and shows this machine's *current* memory status (`audio_df.resources.memory_status`) -- the same check `experiment_run` uses to refuse starting a heavy job under pressure.

In [ ]:
from pathlib import Path

import yaml

from audio_df.experiments.run_a1 import load_config
from audio_df.resources import load_resources, memory_status

cfg = load_config("configs/a1_lfcc_gmm.yaml")
resources_cfg = load_resources("configs/resources.yaml")

print("A1 config summary:")
print(yaml.safe_dump({"features": cfg["features"], "gmm": cfg["gmm"]}, sort_keys=False))

print("Resource limits (configs/resources.yaml):")
print(yaml.safe_dump(resources_cfg["resources"], sort_keys=False))

print("Current memory status:")
print(memory_status())

## Cell 2 - the LFCC pipeline on two real clips (one bona fide, one spoof)

Cheap: one utterance per class. Gated on the dataset actually being present at `configs/data.yaml: dataset.root` -- prints a clear message and skips instead of raising if the download has not happened yet (this notebook's rule, and this task's brief, is that A1's own code never trains or runs on real data by itself; only the user does that from a terminal).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from audio_df.audio import load_audio, preprocess
from audio_df.lfcc_b02 import extract_lfcc_b02

train_manifest_path = Path(cfg["data"]["train"])
if not train_manifest_path.exists():
    print(f"{train_manifest_path} not found -- run scripts/prepare_data.py first. Skipping cell 2.")
else:
    train_df = pd.read_csv(train_manifest_path)
    bona_row = train_df[train_df["label"] == "bonafide"].iloc[0]
    spoof_row = train_df[train_df["label"] == "spoof"].iloc[0]

    fig, axes = plt.subplots(2, 2, figsize=(11, 6))
    for ax_row, row, title in ((axes[0], bona_row, "bona fide"), (axes[1], spoof_row, "spoof")):
        x, sr = load_audio(row["path"], target_sr=cfg["preprocessing"]["sample_rate"])
        x = preprocess(x, sr, silence_mode=cfg["preprocessing"]["silence_mode"], crop_seconds=None)
        feats = extract_lfcc_b02(x, sr=sr, cfg=cfg["features"])

        ax_row[0].plot(x)
        ax_row[0].set_title(f"{title}: waveform ({row['utt_id']})")
        im = ax_row[1].imshow(feats.T, aspect="auto", origin="lower", cmap="magma")
        ax_row[1].set_title(f"{title}: LFCC (60 x {feats.shape[0]} frames)")
        ax_row[1].set_ylabel("static | delta | delta-delta")
    fig.tight_layout()
    plt.show()

## Cell 3 - the 200-clip timing run

Calls exactly the function `run_a1.py timing` uses (`timing_run` over `extract_frames_for_row`), so the notebook and the CLI never drift apart. Read the printed extrapolation as a **lower bound**: this is a fanless machine that throttles under sustained load.

In [ ]:
from audio_df.experiments.run_a1 import extract_frames_for_row
from audio_df.resources import format_timing, timing_run

if not train_manifest_path.exists():
    print("Dataset not present -- skipping the timing run.")
else:
    n = min(200, len(train_df))
    sample_rows = list(train_df.head(n).itertuples(index=False))
    result = timing_run(
        lambda row: extract_frames_for_row(row, cfg),
        sample_rows,
        n=len(sample_rows),
        total_items=len(train_df),
        cfg=resources_cfg,
    )
    print(format_timing(result))

## Cell 4 - the full run: exact terminal commands

Run these in a terminal, not in this notebook -- training is a multi-hour job on this machine and a notebook kernel is not the place for it. `caffeinate -i` stops the laptop from sleeping mid-run; it does not prevent thermal throttling.

In [ ]:
print("# 1) train both GMMs on raw TRAIN, then score raw dev+eval:")
print("caffeinate -i uv run python -m audio_df.experiments.run_a1 train --exp e1")
print()
print("# 2) if interrupted (Ctrl-C, memory pressure, crash), resume the same run:")
print("caffeinate -i uv run python -m audio_df.experiments.run_a1 train --exp e1 --resume")
print()
print("# 3) trimmed diagnostic: score the RAW-trained GMMs against trimmed eval + In-the-Wild")
print("#    (there is no trimmed *training* for A1 -- see run_a1.py's module docstring):")
print(
    "uv run python -m audio_df.experiments.run_a1 score "
    "--model-from experiments/E-<date>-e1-a1-raw --exp e1 "
    "--silence-mode trimmed --testsets eval,itw"
)
print()
print("# 4) turn score files into metrics.json (never inside run_a1.py itself):")
print("uv run python scripts/score_experiment.py experiments/E-<date>-e1-a1-raw")
print(
    "uv run python scripts/score_experiment.py experiments/E-<date>-e1-a1-trimmed "
    "--threshold-from experiments/E-<date>-e1-a1-raw"
)
print()
print("# 5) regenerate the E1...E6 summary tables:")
print("uv run python scripts/make_tables.py")

## Cell 5 - after the run: scores, histograms, metrics, the literature reference

Gated on `experiments/` actually containing a finished A1 run -- set `exp_id` below to whatever your terminal run produced.

In [ ]:
exp_id = None  # e.g. "E-20261001-e1-a1-raw" once you have run the terminal commands above

if exp_id is None:
    print("Set exp_id above to an experiments/<exp_id> directory produced by the terminal run.")
else:
    exp_dir = Path("experiments") / exp_id
    eval_scores = pd.read_csv(exp_dir / "scores" / "eval.csv")

    fig, ax = plt.subplots(figsize=(7, 4))
    for label, color in (("bonafide", "tab:blue"), ("spoof", "tab:red")):
        subset = eval_scores.loc[eval_scores["label"] == label, "score"]
        ax.hist(subset, bins=40, alpha=0.6, label=label, color=color)
    ax.set_xlabel("score (higher = more bona fide)")
    ax.legend()
    ax.set_title(f"{exp_id}: eval score histogram by class")
    plt.show()

    if "attack_id" in eval_scores.columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        spoof_only = eval_scores[eval_scores["label"] == "spoof"]
        spoof_only.boxplot(column="score", by="attack_id", ax=ax)
        ax.set_title(f"{exp_id}: eval spoof score by attack (A07-A19)")
        plt.suptitle("")
        plt.show()

    metrics_path = exp_dir / "metrics.json"
    if metrics_path.exists():
        import json

        metrics = json.loads(metrics_path.read_text())
        print("eval metrics (this run):", metrics["testsets"].get("eval"))
    else:
        print(f"{metrics_path} not found -- run scripts/score_experiment.py first.")

    ref = cfg["literature_reference"]
    print(
        f"\nLITERATURE reference (never compared directly with the run above): "
        f"{ref['source']}, {ref['scenario']} {ref['split']}: "
        f"EER={ref['eer_percent']}%, min t-DCF={ref['min_tdcf']} (tag: {ref['tag']})"
    )